<a href="https://colab.research.google.com/github/ms914/3DKWM_server/blob/main/kwm_cli/KWM_CLI_Mehrfachbindungen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://share.gemini.google/vVXC6mxD25Cp

In [2]:
!pip install clifford

In [1]:
code = """
from fastapi import FastAPI
import numpy as np
from clifford.g3c import *
import uvicorn

# --- PORT-KILLER (Sorgt dafür, dass Colab den Port 8000 freigibt) ---
def kill_port_8000():
    try:
        import subprocess
        result = subprocess.run(["lsof", "-t", "-i:8000"], capture_output=True, text=True)
        pids = result.stdout.strip().split()
        for pid in pids:
            if pid:
                os.kill(int(pid), 9)
        time.sleep(0.5)
    except:
        pass

kill_port_8000()


app = FastAPI()



WELT_ZUSTAND = {}

# --- CGA MATHEMATIK-ENGINE ---
def cga_punkt(x, y, z):
    return up(x*e1 + y*e2 + z*e3)

def berechne_richtungs_rotor(v_start, v_ziel):
    \"\"\"Bringt Vektor v_start zur Deckung mit v_ziel\"\"\"
    v_s = (v_start[0]*e1 + v_start[1]*e2 + v_start[2]*e3).normal()
    v_z = (v_ziel[0]*e1 + v_ziel[1]*e2 + v_ziel[2]*e3).normal()
    gp = v_z * v_s
    return (1 + gp).normal()

def berechne_axialen_rotor(achse_vec, winkel_rad):
    \"\"\"Erzeugt eine Drehung um eine feste Achse (Bivektor-Exponentialform)\"\"\"
    B = (achse_vec[0]*e1 + achse_vec[1]*e2 + achse_vec[2]*e3).normal() * (e1*e2*e3)
    return np.cos(winkel_rad / 2) - np.sin(winkel_rad / 2) * B

# --- PARAMETER ---
# --- PARAMETER ERWEITERT ---
ELEMENT_PROFIL = {
    "H":  {"geo": "KUGEL",     "valenzelektronen": 1, "abstand": 1.10},
    "C":  {"geo": "TETRAEDER", "valenzelektronen": 4, "abstand": 1.25},
    "O":  {"geo": "TETRAEDER", "valenzelektronen": 6, "abstand": 1.20}, # O-H Abstand ca. 1.20 in unserem Gitter
}

def hole_lokale_wolken(symbol):
    if symbol == "H": return [[0.0, 0.0, 0.0]]
    if symbol in ["C", "O"]:
        # Beide nutzen die tetraedrischen Achsen
        return [
            [1/np.sqrt(3), 1/np.sqrt(3), 1/np.sqrt(3)],
            [-1/np.sqrt(3), -1/np.sqrt(3), 1/np.sqrt(3)],
            [-1/np.sqrt(3), 1/np.sqrt(3), -1/np.sqrt(3)],
            [1/np.sqrt(3), -1/np.sqrt(3), -1/np.sqrt(3)]
        ]
    return [[0.0, 0.0, 0.0]]

@app.get("/status")
def get_status():
    return WELT_ZUSTAND

@app.post("/erzeuge")
def erzeuge(daten: dict):
    symbol = daten["symbol"]
    uid = f"{symbol}_{len(WELT_ZUSTAND) + 1}"

    # Valenzelektronen auf die 4 Wolken verteilen
    wolken = []
    lokale_achsen = hole_lokale_wolken(symbol)

    if symbol == "O":
        # Sauerstoff: 6 Elektronen -> zwei Wolken haben 2 (voll), zwei haben 1 (reaktiv)
        elektronen_verteilung = [2, 2, 1, 1]
    elif symbol == "C":
        elektronen_verteilung = [1, 1, 1, 1]
    else: # H
        elektronen_verteilung = [1]

    WELT_ZUSTAND[uid] = {
        "symbol": symbol,
        "position": (np.random.rand(3) - 0.5 * 2).tolist(),
        "rotor_bivektor": [0.0, 0.0, 0.0],
        "wolken": [{"id": i, "elektronen": e, "lokal": v} for i, (e, v) in enumerate(zip(elektronen_verteilung, lokale_achsen))]
    }
    return {"status": "success", "uid": uid}

@app.post("/binde")
def binde(daten: dict):
    if daten["atom_A_id"] not in WELT_ZUSTAND or daten["atom_B_id"] not in WELT_ZUSTAND:
        return {"status": "error", "message": "Eines der Atome existiert nicht. ID prüfen!"}

    atom_A = WELT_ZUSTAND[daten["atom_A_id"]]
    atom_B = WELT_ZUSTAND[daten["atom_B_id"]]

    for w in atom_A["wolken"]:
        if "partner" not in w: w["partner"] = None
    for w in atom_B["wolken"]:
        if "partner" not in w: w["partner"] = None

    bereits_gebunden = sum(1 for w in atom_A["wolken"] if w["partner"] == daten["atom_B_id"])
    ziel_ordnung = bereits_gebunden + 1

    if ziel_ordnung > 3:
        return {"status": "error", "message": "Maximale Bindungsordnung erreicht."}

    # Alte Sortierung nimmt auch existierende nicht bindende Wolken mit 2 Elektronen an
    # freie_A = [w for w in atom_A["wolken"] if w["elektronen"] < 3]
    # freie_B = [w for w in atom_B["wolken"] if w["elektronen"] < 3]

    # Sortiere so, dass Wolken mit 1 Elektron (Radikale) VOR Wolken mit 2 Elektronen (Paare) kommen
    freie_A = sorted([w for w in atom_A["wolken"] if w["elektronen"] < 3], key=lambda x: x["elektronen"])
    freie_B = sorted([w for w in atom_B["wolken"] if w["elektronen"] < 3], key=lambda x: x["elektronen"])

    if not freie_A or not freie_B:
        return {"status": "error", "message": "Keine freien Wolken verfügbar."}

    # Nutze spezifischen C-H Bindungsabstand aus dem Profil

    if "O" in [atom_A["symbol"], atom_B["symbol"]]:
        abstand = ELEMENT_PROFIL["O"]["abstand"]
    else:
        abstand = ELEMENT_PROFIL["H"]["abstand"] if (atom_A["symbol"] == "H" or atom_B["symbol"] == "H") else ELEMENT_PROFIL["C"]["abstand"]




    # 3. GEOMETRISCHE ACHSE (Translation - Jetzt voll dynamisch)
    lokaler_vektor_A = np.array(freie_A[0]["lokal"])
    if np.linalg.norm(lokaler_vektor_A) == 0:
        achse_A = np.array([1.0, 0.0, 0.0])
    else:
        achse_A = lokaler_vektor_A / np.linalg.norm(lokaler_vektor_A)

    neue_pos_B = np.array(atom_A["position"]) + (achse_A * abstand)
    atom_B["position"] = neue_pos_B.tolist()

    # 4. CGA ROTOR-KASKADE (Hier war der Bug-Fix!)
    ziel_achse_B = -achse_A
    start_achse_B = np.array(freie_B[0]["lokal"])
    if np.linalg.norm(start_achse_B) == 0:
        start_achse_B = np.array([1.0, 0.0, 0.0])
    start_achse_B = start_achse_B / np.linalg.norm(start_achse_B)

    # JETZT REPARIERT: start_achse_B wird korrekt übergeben
    R_final = berechne_richtungs_rotor(start_achse_B, ziel_achse_B)

    if ziel_ordnung == 2:
        R_torsion = berechne_axialen_rotor(achse_A, np.radians(0))
        R_final = R_torsion * R_final
    elif ziel_ordnung == 3:
        R_torsion = berechne_axialen_rotor(achse_A, np.radians(60))
        R_final = R_torsion * R_final

    atom_B["rotor_bivektor"] = [float(R_final[e1*e2]), float(R_final[e2*e3]), float(R_final[e3*e1])]

    # 5. STATUS UPDATEN
    freie_A[0]["elektronen"] = 3
    freie_A[0]["partner"] = daten["atom_B_id"]
    freie_B[0]["elektronen"] = 3
    freie_B[0]["partner"] = daten["atom_A_id"]

    return {"status": "success", "bindung_aktuell": ziel_ordnung}


"""

with open("server.py", "w") as f:
    f.write(code)

import subprocess
import time
subprocess.run(["pkill", "-f", "uvicorn"])
time.sleep(1)
subprocess.Popen(["uvicorn", "server:app", "--port", "8000"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
print("⚡ Reparierter CGA-Server läuft stabil im Hintergrund!")

⚡ Reparierter CGA-Server läuft stabil im Hintergrund!


In [2]:
! pip install rdkit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 49.8 MB/s eta 0:00:00


In [3]:
import requests
import json
from rdkit import Chem

def generiere_mol_lokal(welt_zustand):
    # Generiert das MDL MOL-File direkt im Frontend aus den Statusdaten
    if not welt_zustand:
        return ""

    mol_text = "CGA_Baukasten_Export\n  CGA-Engine 2026\n\n"
    anzahl_atome = len(welt_zustand)

    bindungen = []
    gesehene_paare = set()
    uid_zu_index = {uid: idx + 1 for idx, uid in enumerate(welt_zustand.keys())}

    for uid_A, atom in welt_zustand.items():
        for w in atom.get('wolken', []):
            uid_B = w.get('partner')
            if uid_B and uid_B in welt_zustand:
                paar = tuple(sorted([uid_A, uid_B]))
                if paar not in gesehene_paare:
                    gesehene_paare.add(paar)
                    ordnung = sum(1 for wl in atom['wolken'] if wl.get('partner') == uid_B)
                    bindungen.append((uid_zu_index[uid_A], uid_zu_index[uid_B], ordnung))

    anzahl_bindungen = len(bindungen)
    mol_text += "%3d%3d  0  0  0  0  0  0  0  0999 V2000\n" % (anzahl_atome, anzahl_bindungen)

    for uid, atom in welt_zustand.items():
        pos = atom['position']
        symbol = atom['symbol']
        mol_text += "%10.4f%10.4f%10.4f %-3s 0  0  0  0  0  0  0  0  0  0  0  0\n" % (pos[0], pos[1], pos[2], symbol)

    for b in bindungen:
        mol_text += "%3d%3d%3d  0  0  0  0\n" % (b[0], b[1], b[2])

    mol_text += "M  END\n"
    return mol_text


def lade_smiles_in_backend(smiles_string):
    # Parst einen SMILES-String mit RDKit und sendet die Befehle automatisiert an das Backend
    mol = Chem.MolFromSmiles(smiles_string)
    if mol is None:
        print(f"-> Fehler: Ungueltiger SMILES-String '{smiles_string}'")
        return False

    # Wasserstoff-Atome explizit hinzufügen
    mol = Chem.AddHs(mol)

    print(f"-> Interpretiere SMILES: {smiles_string}")
    print(f"-> Gefunden: {mol.GetNumAtoms()} Atome (inkl. H) und {mol.GetNumBonds()} Bindungen.")

    rdkit_idx_zu_uid = {}

    # 1. Schritt: Alle Atome im Backend erzeugen
    for atom in mol.GetAtoms():
        symbol = atom.GetSymbol()
        rdkit_idx = atom.GetIdx()

        if symbol not in ["C", "H", "O"]:
            print(f"-> Warnung: Element {symbol} wird aktuell nicht unterstuetzt. Ueberspringe.")
            continue

        try:
            res = requests.post("http://127.0.0.1:8000/erzeuge", json={"symbol": symbol}).json()
            if res.get("status") == "success":
                rdkit_idx_zu_uid[rdkit_idx] = res['uid']
            else:
                print(f"-> Fehler beim Erzeugen von {symbol}: {res.get('message')}")
        except:
            print("-> Fehler: Server nicht erreichbar.")
            return False

    # 2. Schritt: Alle Bindungen im Backend knuepfen
    for bond in mol.GetBonds():
        idx_A = bond.GetBeginAtomIdx()
        idx_B = bond.GetEndAtomIdx()

        if idx_A in rdkit_idx_zu_uid and idx_B in rdkit_idx_zu_uid:
            uid_A = rdkit_idx_zu_uid[idx_A]
            uid_B = rdkit_idx_zu_uid[idx_B]

            b_type = bond.GetBondType()
            wiederholungen = 1
            if b_type == Chem.BondType.DOUBLE:
                wiederholungen = 2
            elif b_type == Chem.BondType.TRIPLE:
                wiederholungen = 3

            for _ in range(wiederholungen):
                try:
                    requests.post("http://127.0.0.1:8000/binde", json={"atom_A_id": uid_A, "atom_B_id": uid_B})
                except:
                    print("-> Fehler beim Knuepfen einer Bindung (Server-Verbindung verloren).")
                    return False

    print("-> SMILES erfolgreich geladen und im CGA-Raum platziert!")
    return True


def zeige_welt(welt_zustand):
    print("\n" + "="*95)
    print(" CGA-MOLEKÜL-BAUKASTEN (LEWIS-SCHREIBWEISE)")
    print("="*95)

    if not welt_zustand:
        print(" Keine Atome in der Szene. Nutze 'erzeuge <C/H/O>' oder 'smiles <string>' um zu starten.")
        print("="*95)
        return

    for uid, atom in welt_zustand.items():
        symbol = atom["symbol"]
        pos = atom["position"]
        pos_str = [f"{p:.2f}" for p in pos]

        kwm_symbole = []
        for w in atom["wolken"]:
            if w.get("partner"):
                p_id = w["partner"]
                bindungsstufe = sum(1 for wl in atom["wolken"] if wl.get("partner") == p_id)
                if bindungsstufe == 3: kwm_symbole.append("≡")
                elif bindungsstufe == 2: kwm_symbole.append("═")
                else: kwm_symbole.append("─")
            else:
                if w["elektronen"] == 2: kwm_symbole.append("|")
                else: kwm_symbole.append("•")

        kwm_str = " ".join(kwm_symbole)

        freie_paare = sum(1 for w in atom["wolken"] if w["elektronen"] == 2 and not w.get("partner"))
        radikale = sum(1 for w in atom["wolken"] if w["elektronen"] == 1 and not w.get("partner"))

        lewis_str = "|" * freie_paare + symbol + "•" * radikale
        partner_erwaehnt = set()
        for w in atom["wolken"]:
            p_id = w.get("partner")
            if p_id and p_id not in partner_erwaehnt:
                partner_erwaehnt.add(p_id)
                bindungsstufe = sum(1 for wl in atom["wolken"] if wl.get("partner") == p_id)
                if bindungsstufe == 3: lewis_str += f" ≡{p_id}"
                elif bindungsstufe == 2: lewis_str += f" ═{p_id}"
                else: lewis_str += f" ─{p_id}"

        if freie_paare > 0 and not partner_erwaehnt:
            lewis_str += "|" * freie_paare

        print(f"ID: {uid:<6} | Pos: {pos_str} | KWM: [ {kwm_str} ] | Lewis: {lewis_str}")
    print("="*95)


def cli_baukasten():
    print("CLI-Frontend erfolgreich geladen.")
    aktuelle_welt = {}

    try:
        res = requests.get("http://127.0.0.1:8000/status")
        aktuelle_welt = res.json()
        zeige_welt(aktuelle_welt)
    except:
        zeige_welt({})

    while True:
        eingabe = input("Befehl [erzeuge <S> | binde <id1> <id2> | smiles <string> | export | exit]: ").strip().split()
        if not eingabe: continue
        kommando = eingabe[0].lower()

        if kommando == "exit":
            print("Baukasten beendet. Auf Wiedersehen!")
            break

        elif kommando == "smiles" and len(eingabe) > 1:
            smiles_str = eingabe[1]
            lade_smiles_in_backend(smiles_str)

        elif kommando == "erzeuge" and len(eingabe) > 1:
            symbol = eingabe[1].upper()
            if symbol in ["C", "H", "O"]:
                try:
                    res = requests.post("http://127.0.0.1:8000/erzeuge", json={"symbol": symbol}).json()
                except: print("-> Fehler: Server offline.")
            else: print("-> Ungueltiges Element.")

        elif kommando == "binde" and len(eingabe) > 2:
            try:
                res = requests.post("http://127.0.0.1:8000/binde", json={"atom_A_id": eingabe[1], "atom_B_id": eingabe[2]}).json()
            except: print("-> Fehler: Verbindung fehlgeschlagen.")

        elif kommando == "export":
            if aktuelle_welt:
                mol_data = generiere_mol_lokal(aktuelle_welt)
                with open("molekuel.mol", "w") as f:
                    f.write(mol_data)
                print("-> GENIAL! 'molekuel.mol' wurde direkt im Frontend generiert und gespeichert.")
            else:
                print("-> Fehler: Keine Daten zum Exportieren vorhanden.")
        else:
            print("-> Unbekannter Befehl.")

        try:
            res = requests.get("http://127.0.0.1:8000/status")
            aktuelle_welt = res.json()
            zeige_welt(aktuelle_welt)
        except:
            print(" [Warnung] Synchronisation fehlgeschlagen.")

# CLI direkt ausführen
cli_baukasten()

CLI-Frontend erfolgreich geladen.

 CGA-MOLEKÜL-BAUKASTEN (LEWIS-SCHREIBWEISE)
 Keine Atome in der Szene. Nutze 'erzeuge <C/H/O>' oder 'smiles <string>' um zu starten.
Befehl [erzeuge <S> | binde <id1> <id2> | smiles <string> | export | exit]: smiles C=C
-> Interpretiere SMILES: C=C
-> Gefunden: 6 Atome (inkl. H) und 5 Bindungen.
-> SMILES erfolgreich geladen und im CGA-Raum platziert!

 CGA-MOLEKÜL-BAUKASTEN (LEWIS-SCHREIBWEISE)
ID: C_1    | Pos: ['-0.09', '-0.25', '-0.77'] | KWM: [ ═ ═ ─ ─ ] | Lewis: C ═C_2 ─H_3 ─H_4
ID: C_2    | Pos: ['-0.81', '-0.97', '-0.05'] | KWM: [ ═ ═ ─ ─ ] | Lewis: C ═C_1 ─H_5 ─H_6
ID: H_3    | Pos: ['-0.72', '0.39', '-1.41'] | KWM: [ ─ ] | Lewis: H ─C_1
ID: H_4    | Pos: ['0.55', '-0.88', '-1.41'] | KWM: [ ─ ] | Lewis: H ─C_1
ID: H_5    | Pos: ['-1.45', '-0.33', '-0.69'] | KWM: [ ─ ] | Lewis: H ─C_2
ID: H_6    | Pos: ['-0.17', '-1.60', '-0.69'] | KWM: [ ─ ] | Lewis: H ─C_2
Befehl [erzeuge <S> | binde <id1> <id2> | smiles <string> | export | exit]: export
-> 

In [4]:
!pip install py3Dmol -q

In [5]:
import py3Dmol
import os

def zeige_3d_molekuel_lokal(dateiname="molekuel.mol"):
    # Prüfen, ob das Frontend die Datei bereits erzeugt hat
    if not os.path.exists(dateiname):
        print(f"Fehler: Die Datei '{dateiname}' wurde noch nicht exportiert.")
        print("Bitte gib zuerst im CLI den Befehl 'export' ein.")
        return

    try:
        # Die lokal gespeicherte .mol-Datei einlesen
        with open(dateiname, "r") as f:
            mol_data = f.read()

        if not mol_data.strip():
            print("Die exportierte Datei ist leer.")
            return

        # py3Dmol Viewer initialisieren
        view = py3Dmol.view(width=600, height=400)
        view.addModel(mol_data, "mol")

        # Schickes Rendering: Stick (Stäbchen) + Sphere (Kugeln für die Atome)
        view.setStyle({'stick': {'radius': 0.15}, 'sphere': {'scale': 0.3}})

        # Automatisch auf das Molekül zentrieren und anzeigen
        view.zoomTo()
        view.show()

        print(f"🎉 '{dateiname}' erfolgreich in interaktives 3D-Modell geladen!")

    except Exception as e:
        print(f"Fehler beim Rendern der 3D-Grafik: {e}")

# Die Visualisierung starten
zeige_3d_molekuel_lokal()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

🎉 'molekuel.mol' erfolgreich in interaktives 3D-Modell geladen!
